# 03 — Multi-Agentes: Pipeline Condicional

**Módulo:** EAI_07 — IA Generativa  
**Submódulo:** 05_Agentes  
**Ambiente:** `eai07` (Python 3.11)

---

## O que você vai aprender

- **Roteamento inteligente** — um agente decide qual especialista chamar com base na pergunta
- **Agentes especializados** — cada agente tem ferramentas e system prompt próprios
- **Pipeline condicional** — o fluxo muda dependendo do tipo de tarefa
- **Composição de resultados** — saída de um agente vira contexto do próximo

---

### Arquitetura

```
                    ┌─────────────────────────────────────────┐
  Pergunta  ──────► │           AGENTE ROTEADOR               │
                    │  Classifica: pesquisa | codigo | math   │
                    │  Decide: simples | composto             │
                    └──────────────┬──────────────────────────┘
                                   │
              ┌────────────────────┼────────────────────┐
              ▼                    ▼                    ▼
     ┌─────────────────┐  ┌──────────────────┐  ┌─────────────────┐
     │   ESPECIALISTA  │  │   ESPECIALISTA   │  │   ESPECIALISTA  │
     │    PESQUISA     │  │     CÓDIGO       │  │   MATEMÁTICA    │
     │  buscar_web     │  │  ler_arquivo     │  │   calcular      │
     │  consultar_curso│  │  listar_arquivos │  │   (avançado)    │
     └────────┬────────┘  │  gerar_codigo    │  └────────┬────────┘
              │           └────────┬─────────┘           │
              │                    │                      │
              └────────────────────┼──────────────────────┘
                                   ▼
                          ┌─────────────────┐
                          │   SINTETIZADOR  │  (apenas tarefas compostas)
                          │  combina saídas │
                          └────────┬────────┘
                                   │
                                   ▼
                              Resposta final
```

## Setup

In [4]:
import sys, os, json, math, re
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

sys.path.append(os.path.abspath('..'))
load_dotenv('../.env')

from shared.tool_runner import executar_com_tools, ToolRunner

# llm usado diretamente pelo roteador e sintetizador
llm = OpenAI(
    api_key=os.getenv('DEEPSEEK_API_KEY'),
    base_url='https://api.deepseek.com'
)
LLM_MODEL    = os.getenv('LLM_MODEL', 'deepseek-chat')
PROJETO_BASE = os.path.abspath('../..')
EAI07_BASE   = os.path.abspath('..')

print(f'LLM     : {LLM_MODEL}')
print(f'Projeto : {PROJETO_BASE}')


LLM     : deepseek-chat
Projeto : C:\Users\Jorge Maques\Documents\Especialista_em_AI


---
## 1. Ferramentas dos Especialistas

Cada especialista tem um conjunto de ferramentas próprio.  
Reutilizamos as implementações do notebook 02 — copiadas para este ser independente.

In [5]:
import requests
from bs4 import BeautifulSoup

# ── Ferramentas de Pesquisa ───────────────────────────────────────────────

def buscar_web(query: str, max_resultados: int = 4) -> str:
    """Busca no DuckDuckGo e retorna título + snippet dos resultados."""
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0'}
    try:
        url  = f'https://html.duckduckgo.com/html/?q={requests.utils.quote(query)}'
        resp = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(resp.text, 'html.parser')
        resultados = []
        for item in soup.select('.result')[:max_resultados]:
            t = item.select_one('.result__title')
            s = item.select_one('.result__snippet')
            if t and s:
                resultados.append(f'• {t.get_text(strip=True)}\n  {s.get_text(strip=True)}')
        return f'Resultados para "{query}":\n\n' + '\n\n'.join(resultados) if resultados else f'Sem resultados para: {query}'
    except Exception as e:
        return f'Erro na busca: {e}'


def consultar_curso(pergunta: str, modulo: str = '') -> str:
    """Busca informações nos AGENT_CONTEXT.md do curso."""
    palavras   = [p for p in pergunta.lower().split() if len(p) > 3]
    ignorar    = {'.git', 'venv', '.venv', '__pycache__', '.ipynb_checkpoints'}
    resultados = []
    for raiz, dirs, arquivos in os.walk(PROJETO_BASE):
        dirs[:] = [d for d in dirs if d not in ignorar]
        if 'AGENT_CONTEXT.md' not in arquivos:
            continue
        partes = raiz.replace('\\', '/').split('/')
        mod    = next((p for p in partes if p.startswith('EAI_')), '')
        if modulo and not mod.startswith(modulo):
            continue
        with open(os.path.join(raiz, 'AGENT_CONTEXT.md'), encoding='utf-8') as f:
            for linha in f:
                hits = sum(1 for p in palavras if p in linha.lower())
                if hits >= min(2, len(palavras)):
                    resultados.append((hits, f'[{mod}] {linha.strip()}'))
    if not resultados:
        return f'Nenhuma informação encontrada para: {pergunta}'
    resultados.sort(key=lambda x: x[0], reverse=True)
    return '\n'.join(r for _, r in resultados[:8])


# ── Ferramentas de Código ─────────────────────────────────────────────────

def ler_arquivo(caminho: str, max_linhas: int = 80) -> str:
    """Lê conteúdo de arquivo do projeto (.py, .md, .txt)."""
    ext_ok = {'.py', '.md', '.txt', '.json', '.yaml', '.toml'}
    for base in [EAI07_BASE, PROJETO_BASE]:
        c = Path(base) / caminho
        if c.exists():
            break
    else:
        return f'Arquivo não encontrado: {caminho}'
    if c.suffix not in ext_ok:
        return f'Extensão não suportada: {c.suffix}'
    linhas = c.read_text(encoding='utf-8').splitlines()
    texto  = '\n'.join(linhas[:max_linhas])
    aviso  = f'\n\n[... +{len(linhas)-max_linhas} linhas omitidas]' if len(linhas) > max_linhas else ''
    return f'# {c.name} ({len(linhas)} linhas)\n\n{texto}{aviso}'


def listar_arquivos(caminho: str = '', extensao: str = '') -> str:
    """Lista arquivos de um diretório do projeto."""
    base    = Path(EAI07_BASE) / caminho if caminho else Path(EAI07_BASE)
    ignorar = {'.git', '__pycache__', '.ipynb_checkpoints', 'venv', '.venv'}
    if not base.exists():
        return f'Diretório não encontrado: {caminho}'
    arqs = [
        str(item.relative_to(Path(EAI07_BASE)))
        for item in sorted(base.rglob('*'))
        if item.is_file()
        and not any(p in item.parts for p in ignorar)
        and (not extensao or item.suffix == extensao)
    ]
    return f'{len(arqs)} arquivo(s):\n' + '\n'.join(arqs[:30]) if arqs else 'Nenhum arquivo encontrado.'


def gerar_codigo(descricao: str, contexto: str = '') -> str:
    """Usa LLM especializado para gerar código Python."""
    prompt = descricao + (f'\n\nContexto: {contexto}' if contexto else '')
    resp   = llm.chat.completions.create(
        model       = LLM_MODEL,
        messages    = [
            {'role': 'system', 'content': 'Você é especialista em Python. Gere código limpo e funcional. Retorne APENAS o código.'},
            {'role': 'user',   'content': prompt},
        ],
        temperature = 0.2,
    )
    codigo = resp.choices[0].message.content.strip()
    return re.sub(r'^```python\s*|^```\s*|\s*```$', '', codigo, flags=re.MULTILINE).strip()


# ── Ferramentas de Matemática ─────────────────────────────────────────────

def calcular(expressao: str) -> str:
    """Avalia expressões matemáticas com segurança."""
    permitidos = {
        'sqrt': math.sqrt, 'log': math.log, 'log10': math.log10,
        'sin': math.sin,   'cos': math.cos, 'tan': math.tan,
        'pi': math.pi,     'e': math.e,     'abs': abs, 'round': round,
    }
    try:
        return str(eval(expressao, {'__builtins__': {}}, permitidos))
    except Exception as ex:
        return f'Erro: {ex}'


print('Ferramentas carregadas: buscar_web, consultar_curso, ler_arquivo, listar_arquivos, gerar_codigo, calcular')

Ferramentas carregadas: buscar_web, consultar_curso, ler_arquivo, listar_arquivos, gerar_codigo, calcular


---
## 2. Schemas das Ferramentas

In [6]:
# Schemas por especialista — cada um recebe só o que precisa

SCHEMAS_PESQUISA = [
    {'type': 'function', 'function': {'name': 'buscar_web',      'description': 'Busca informações atuais na web via DuckDuckGo.',            'parameters': {'type': 'object', 'properties': {'query':    {'type': 'string'}, 'max_resultados': {'type': 'integer'}}, 'required': ['query']}}},
    {'type': 'function', 'function': {'name': 'consultar_curso', 'description': 'Busca informações técnicas nos módulos EAI_01 a EAI_08.',    'parameters': {'type': 'object', 'properties': {'pergunta': {'type': 'string'}, 'modulo':          {'type': 'string'}},  'required': ['pergunta']}}},
]

SCHEMAS_CODIGO = [
    {'type': 'function', 'function': {'name': 'listar_arquivos', 'description': 'Lista arquivos do projeto. Use caminho="shared" para o shared/.', 'parameters': {'type': 'object', 'properties': {'caminho': {'type': 'string'}, 'extensao': {'type': 'string'}}, 'required': []}}},
    {'type': 'function', 'function': {'name': 'ler_arquivo',     'description': 'Lê conteúdo de arquivo do projeto (.py, .md, .txt).',            'parameters': {'type': 'object', 'properties': {'caminho': {'type': 'string'}, 'max_linhas': {'type': 'integer'}}, 'required': ['caminho']}}},
    {'type': 'function', 'function': {'name': 'gerar_codigo',    'description': 'Gera código Python para uma tarefa específica.',                 'parameters': {'type': 'object', 'properties': {'descricao': {'type': 'string'}, 'contexto': {'type': 'string'}}, 'required': ['descricao']}}},
]

SCHEMAS_MATEMATICA = [
    {'type': 'function', 'function': {'name': 'calcular', 'description': 'Avalia expressões matemáticas. Ex: "sqrt(144)", "pi * 5**2".', 'parameters': {'type': 'object', 'properties': {'expressao': {'type': 'string'}}, 'required': ['expressao']}}},
]

FUNCOES_PESQUISA   = {'buscar_web': buscar_web, 'consultar_curso': consultar_curso}
FUNCOES_CODIGO     = {'listar_arquivos': listar_arquivos, 'ler_arquivo': ler_arquivo, 'gerar_codigo': gerar_codigo}
FUNCOES_MATEMATICA = {'calcular': calcular}

print('Schemas definidos.')

Schemas definidos.


---
## 3. Agente Roteador

O roteador classifica a pergunta em um de três tipos e decide se a tarefa
é simples (um especialista) ou composta (dois especialistas + síntese).

Retorna um dicionário estruturado com a decisão de roteamento.

In [7]:
SYSTEM_ROTEADOR = """\
Você é um roteador de tarefas. Analise a pergunta e classifique qual(is) especialista(s) deve(m) responder.

Especialistas disponíveis:
- pesquisa : busca na web ou nos módulos do curso EAI_01-EAI_08
- codigo   : leitura de arquivos, estrutura do projeto, geração de código Python
- matematica: cálculos numéricos, expressões matemáticas

Responda APENAS com JSON válido, sem texto antes ou depois:
{
  "tipo": "simples" | "composto",
  "especialistas": ["pesquisa"] | ["codigo"] | ["matematica"] | ["pesquisa", "codigo"] | etc,
  "justificativa": "<1 frase>"
}

Use "composto" apenas quando a pergunta realmente precisar de dois tipos diferentes.
"""


def rotear(pergunta: str, verbose: bool = True) -> dict:
    """
    Classifica a pergunta e retorna a decisão de roteamento.
    Retorna: {tipo, especialistas, justificativa}
    """
    resp  = llm.chat.completions.create(
        model       = LLM_MODEL,
        messages    = [
            {'role': 'system', 'content': SYSTEM_ROTEADOR},
            {'role': 'user',   'content': pergunta},
        ],
        temperature = 0.0,
    )
    texto = resp.choices[0].message.content.strip()
    texto = re.sub(r'^```json\s*|^```\s*|\s*```$', '', texto, flags=re.MULTILINE).strip()

    try:
        decisao = json.loads(texto)
    except json.JSONDecodeError:
        # Fallback: tudo vai para pesquisa
        decisao = {'tipo': 'simples', 'especialistas': ['pesquisa'], 'justificativa': 'fallback'}

    if verbose:
        print(f'  [ROTEADOR] tipo={decisao["tipo"]} | especialistas={decisao["especialistas"]}')
        print(f'             justificativa: {decisao["justificativa"]}')

    return decisao


# Teste do roteador isolado
perguntas_teste = [
    'Quanto é a raiz de 225 dividido por 3?',
    'Quais são os arquivos Python no shared/ do EAI_07?',
    'Busque notícias sobre o modelo GPT-5',
    'Leia o llm_factory.py e gere um exemplo de uso. Depois calcule quantas linhas ele tem.',
]

print('Testando roteador:\n')
for p in perguntas_teste:
    print(f'Pergunta: {p}')
    rotear(p)
    print()

Testando roteador:

Pergunta: Quanto é a raiz de 225 dividido por 3?
  [ROTEADOR] tipo=simples | especialistas=['matematica']
             justificativa: A pergunta envolve apenas um cálculo matemático simples (raiz quadrada e divisão).

Pergunta: Quais são os arquivos Python no shared/ do EAI_07?
  [ROTEADOR] tipo=simples | especialistas=['codigo']
             justificativa: A pergunta pede para listar arquivos em um diretório específico do projeto, o que é uma tarefa de leitura de estrutura de arquivos.

Pergunta: Busque notícias sobre o modelo GPT-5
  [ROTEADOR] tipo=simples | especialistas=['pesquisa']
             justificativa: A tarefa é uma solicitação de busca por informações atualizadas na web.

Pergunta: Leia o llm_factory.py e gere um exemplo de uso. Depois calcule quantas linhas ele tem.
  [ROTEADOR] tipo=composto | especialistas=['codigo', 'matematica']
             justificativa: A pergunta pede primeiro a leitura de um arquivo e geração de um exemplo (codigo) e depois 

---
## 4. Agentes Especialistas

Cada especialista é um `ToolRunner` com system prompt e ferramentas próprias.
Pode receber contexto extra (saída de outro especialista) para tarefas compostas.

In [8]:
def criar_especialista_pesquisa() -> ToolRunner:
    runner = ToolRunner(
        system="""\
Você é um especialista em pesquisa. Use buscar_web para informações atuais da internet
e consultar_curso para informações sobre os módulos EAI_01-EAI_08.
Seja preciso, cite as fontes e seja conciso.""",
        verbose=True
    )
    for s, f in zip(SCHEMAS_PESQUISA, FUNCOES_PESQUISA.values()):
        runner.registrar(s, f)
    return runner


def criar_especialista_codigo() -> ToolRunner:
    runner = ToolRunner(
        system="""\
Você é um especialista em código Python. Use listar_arquivos e ler_arquivo para
explorar o projeto, e gerar_codigo para criar implementações.
Seja técnico, mostre código quando relevante.""",
        verbose=True
    )
    for s, f in zip(SCHEMAS_CODIGO, FUNCOES_CODIGO.values()):
        runner.registrar(s, f)
    return runner


def criar_especialista_matematica() -> ToolRunner:
    runner = ToolRunner(
        system="""\
Você é um especialista em matemática. Use calcular para todas as operações numéricas.
Mostre os passos do raciocínio e o resultado final claramente.""",
        verbose=True
    )
    runner.registrar(SCHEMAS_MATEMATICA[0], calcular)
    return runner


# Mapa: nome → factory
ESPECIALISTAS = {
    'pesquisa'   : criar_especialista_pesquisa,
    'codigo'     : criar_especialista_codigo,
    'matematica' : criar_especialista_matematica,
}

print('Especialistas definidos:', list(ESPECIALISTAS.keys()))

Especialistas definidos: ['pesquisa', 'codigo', 'matematica']


---
## 5. Pipeline Condicional

O pipeline orquestra o fluxo completo:

```
rotear(pergunta)
    │
    ├── tipo == 'simples'  → especialista único → resposta
    │
    └── tipo == 'composto' → especialista_1 → resultado_1
                          → especialista_2(contexto=resultado_1) → resultado_2
                          → sintetizador(resultado_1, resultado_2) → resposta final
```

In [9]:
def sintetizar(pergunta: str, resultados: dict) -> str:
    """
    Combina resultados de múltiplos especialistas em uma resposta coerente.
    resultados: {nome_especialista: resposta}
    """
    partes = '\n\n'.join(
        f'=== Especialista {nome.upper()} ===\n{resp}'
        for nome, resp in resultados.items()
    )
    prompt = f"""Pergunta original: {pergunta}

Resultados dos especialistas:
{partes}

Sintetize os resultados acima em uma resposta única, coerente e completa.
Integre as informações sem repetição desnecessária."""

    resp = llm.chat.completions.create(
        model    = LLM_MODEL,
        messages = [
            {'role': 'system', 'content': 'Você sintetiza resultados de múltiplos especialistas em uma resposta clara e integrada.'},
            {'role': 'user',   'content': prompt},
        ],
    )
    return resp.choices[0].message.content


def executar_pipeline(
    pergunta : str,
    verbose  : bool = True,
) -> str:
    """
    Pipeline completo: rotear → especialistas → [síntese].

    Fluxo condicional:
    - simples  → 1 especialista, resposta direta
    - composto → N especialistas em sequência, síntese final
    """
    sep = '─' * 55
    if verbose:
        print(f'\n{sep}')
        print(f'PIPELINE | {pergunta[:60]}')
        print(sep)

    # ── Etapa 1: Roteamento ───────────────────────────────────
    decisao = rotear(pergunta, verbose=verbose)

    especialistas_selecionados = [
        nome for nome in decisao['especialistas']
        if nome in ESPECIALISTAS
    ]

    if not especialistas_selecionados:
        especialistas_selecionados = ['pesquisa']  # fallback

    # ── Etapa 2: Execução dos especialistas ───────────────────
    resultados = {}
    contexto_acumulado = ''

    for nome in especialistas_selecionados:
        if verbose:
            print(f'\n  → Especialista: {nome.upper()}')

        agente = ESPECIALISTAS[nome]()

        # Enriquece a pergunta com contexto dos especialistas anteriores
        pergunta_enriquecida = pergunta
        if contexto_acumulado:
            pergunta_enriquecida = (
                f'{pergunta}\n\n'
                f'Contexto já obtido por outro especialista:\n{contexto_acumulado}'
            )

        resultado = agente.perguntar(pergunta_enriquecida)
        resultados[nome]       = resultado
        contexto_acumulado    += f'\n[{nome}]: {resultado}'

    # ── Etapa 3: Síntese condicional ──────────────────────────
    if len(resultados) == 1:
        # Simples: retorna direto sem síntese
        resposta_final = list(resultados.values())[0]
    else:
        # Composto: sintetiza os resultados
        if verbose:
            print('\n  → Sintetizador')
        resposta_final = sintetizar(pergunta, resultados)

    return resposta_final


print('Pipeline condicional definido.')

Pipeline condicional definido.


---
## 6. Testes do Pipeline

In [10]:
# Teste 1: simples → matemática
print('TESTE 1 — Pipeline simples (matemática)')
resp = executar_pipeline('Quanto é log10(1000) + sqrt(81)?')
print(f'\n🤖 {resp}')

TESTE 1 — Pipeline simples (matemática)

───────────────────────────────────────────────────────
PIPELINE | Quanto é log10(1000) + sqrt(81)?
───────────────────────────────────────────────────────
  [ROTEADOR] tipo=simples | especialistas=['matematica']
             justificativa: A pergunta envolve apenas cálculos matemáticos (logaritmo e raiz quadrada).

  → Especialista: MATEMATICA
[iter 1] 1 tool call(s) — OpenAI
  -> calcular({'expressao': 'log10(1000)'})
     = 3.0
[iter 2] 1 tool call(s) — OpenAI
  -> calcular({'expressao': 'sqrt(81)'})
     = 9.0
[iter 3] 1 tool call(s) — OpenAI
  -> calcular({'expressao': '3 + 9'})
     = 12
[iter 4] Resposta final

🤖 **Resposta:**
log₁₀(1000) + √81 = 3 + 9 = **12**


In [11]:
# Teste 2: simples → código
print('TESTE 2 — Pipeline simples (código)')
resp = executar_pipeline('Quais arquivos Python existem no diretório shared/?')
print(f'\n🤖 {resp}')

TESTE 2 — Pipeline simples (código)

───────────────────────────────────────────────────────
PIPELINE | Quais arquivos Python existem no diretório shared/?
───────────────────────────────────────────────────────
  [ROTEADOR] tipo=simples | especialistas=['codigo']
             justificativa: A pergunta pede para listar arquivos em um diretório específico do projeto, o que é uma tarefa de leitura de estrutura de arquivos.

  → Especialista: CODIGO
[iter 1] 1 tool call(s) — OpenAI
  -> listar_arquivos({'caminho': 'shared', 'extensao': '.py'})
     = 2 arquivo(s):
shared\llm_factory.py
shared\tool_runner.py
[iter 2] 1 tool call(s) — OpenAI
  -> ler_arquivo({'caminho': 'shared/llm_factory.py'})
     = # llm_factory.py (231 linhas)

"""
shared/llm_factory.py
Factory provider-agnóstica compartilhada por todos os módulos do EAI_07.

Como usar nos notebooks:
    import sys, os
    sys.path.append(os.path.abspath('..'))          # aponta para EAI_07/
    from shared.llm_factory import chat, get

In [12]:
# Teste 3: simples → pesquisa
print('TESTE 3 — Pipeline simples (pesquisa no curso)')
resp = executar_pipeline('Quais técnicas de chunking foram implementadas no submódulo 03_RAG do EAI_07?')
print(f'\n🤖 {resp}')

TESTE 3 — Pipeline simples (pesquisa no curso)

───────────────────────────────────────────────────────
PIPELINE | Quais técnicas de chunking foram implementadas no submódulo 
───────────────────────────────────────────────────────
  [ROTEADOR] tipo=simples | especialistas=['pesquisa']
             justificativa: A pergunta pede informações específicas sobre o conteúdo de um submódulo do curso, o que requer consulta ao material didático.

  → Especialista: PESQUISA
[iter 1] 1 tool call(s) — OpenAI
  -> consultar_curso({'pergunta': 'Quais técnicas de chunking foram implementadas no submódulo 03_RAG do EAI_07?', 'modulo': 'EAI_07'})
     = [EAI_07_AI_Generative] | 03_rag_basico | Embedding, FAISS IndexFlatIP, chunking por seção, enriquecimento |
[EAI_07_AI_Generative] **Técnicas implementadas**:
[EAI_07_AI_Generative] - Submódulo: 03_RAG
[EAI_07_AI_Generative] - Chunking: por seções markdown (mesmo padrão do 03_RAG)
[iter 2] Resposta final

🤖 Com base na consulta ao curso, posso fornecer

In [13]:
# Teste 4: composto → código + matemática
print('TESTE 4 — Pipeline composto (código + matemática)')
resp = executar_pipeline(
    'Liste os arquivos do shared/ e conte quantas linhas tem o llm_factory.py. '
    'Depois calcule a média de linhas por arquivo .py do shared/.'
)
print(f'\n🤖 {resp}')

TESTE 4 — Pipeline composto (código + matemática)

───────────────────────────────────────────────────────
PIPELINE | Liste os arquivos do shared/ e conte quantas linhas tem o ll
───────────────────────────────────────────────────────
  [ROTEADOR] tipo=composto | especialistas=['codigo', 'matematica']
             justificativa: A tarefa requer listagem e contagem de linhas de arquivos (codigo) e depois um cálculo de média (matematica).

  → Especialista: CODIGO
[iter 1] 1 tool call(s) — OpenAI
  -> listar_arquivos({'caminho': 'shared'})
     = 3 arquivo(s):
shared\llm_factory.py
shared\requirements.txt
shared\tool_runner.py
[iter 2] 1 tool call(s) — OpenAI
  -> ler_arquivo({'caminho': 'shared/llm_factory.py'})
     = # llm_factory.py (231 linhas)

"""
shared/llm_factory.py
Factory provider-agnóstica compartilhada por todos os módulos do EAI_07.

Como usar nos notebooks:
    import sys, os
    sys.path.append(os.path.abspath('..'))          # aponta para EAI_07/
    from shared.llm_fac

---
## 7. Inspecionando o Fluxo Condicional

Visualizamos quais caminhos o roteador escolhe para diferentes tipos de pergunta.

In [14]:
perguntas_variadas = [
    'Quanto é 2 elevado a 10?',
    'O que é BM25 e como foi usado no EAI_07?',
    'Gere uma função Python para calcular fibonacci',
    'Leia o tool_runner.py, explique a função _parse_dsml e calcule quantas linhas ela tem',
    'Busque na web o que é o padrão ReAct em agentes de IA',
]

print('Mapa de roteamento:\n')
print(f'{"Pergunta":<60} {"Tipo":<10} {"Especialistas"}')
print('-' * 90)

for p in perguntas_variadas:
    d = rotear(p, verbose=False)
    print(f'{p[:58]:<60} {d["tipo"]:<10} {d["especialistas"]}')

Mapa de roteamento:

Pergunta                                                     Tipo       Especialistas
------------------------------------------------------------------------------------------
Quanto é 2 elevado a 10?                                     simples    ['matematica']
O que é BM25 e como foi usado no EAI_07?                     composto   ['pesquisa', 'codigo']
Gere uma função Python para calcular fibonacci               simples    ['codigo']
Leia o tool_runner.py, explique a função _parse_dsml e cal   composto   ['codigo', 'matematica']
Busque na web o que é o padrão ReAct em agentes de IA        simples    ['pesquisa']


---
## Resumo

| Componente | Responsabilidade |
|---|---|
| **Roteador** | Classifica pergunta → tipo + especialistas, retorna JSON |
| **Especialista Pesquisa** | `buscar_web` + `consultar_curso` |
| **Especialista Código** | `listar_arquivos` + `ler_arquivo` + `gerar_codigo` |
| **Especialista Matemática** | `calcular` |
| **Sintetizador** | Combina saídas de múltiplos especialistas |
| **Pipeline** | Orquestra: rotear → especialistas → síntese condicional |

### Fluxo condicional

```python
if tipo == 'simples':
    return especialista.perguntar(pergunta)          # direto
else:
    for especialista in especialistas:
        resultado = especialista.perguntar(pergunta + contexto_acumulado)
    return sintetizar(resultados)                    # síntese
```

### Boas práticas

- **Roteador com `temperature=0.0`** — classificação deve ser determinística
- **Contexto acumulado** — cada especialista recebe a saída dos anteriores
- **Síntese só quando necessário** — evita custo extra em tarefas simples
- **Fallback no roteador** — se JSON inválido, vai para pesquisa por padrão